In [9]:
import numpy as np
import pandas as pd
import matplotlib as mlt
import seaborn as sb
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
from concurrent.futures import ThreadPoolExecutor



load_dotenv()

True

In [10]:
DATABASE_URL = os.getenv("DATABASE_URL")
sql_engine = create_engine(DATABASE_URL)

In [11]:
# data = pd.read_csv("activity.csv") # old implementation ; but now i am directly pooling data from database
# data pooling staement section started here
select_activity_stm = "SELECT * FROM activity"
select_body_battery_activity_events_stm = "SELECT * FROM body_battery_activity_events"
select_body_battery_sample_stm = "SELECT * FROM body_battery_sample"
select_goals_stm = "SELECT * FROM goals"
select_heart_rate_sample_stm = "SELECT * FROM heart_rate_sample"
select_heart_rate_values_stm = "SELECT * FROM heart_rate_values"
select_kilocalories_stm = "SELECT * FROM kilocalories"
select_overall_sleep_stm = "SELECT * FROM overall_sleep"
select_sleep_stm = "SELECT * FROM sleep"
select_sleep_level_events_stm = "SELECT * FROM sleep_level_events"
select_steps_stm = "SELECT * FROM steps"
select_stress_stm = "SELECT * FROM stress"
select_stress_level_sample_stm = "SELECT * FROM stress_level_sample"
# data pooling statement section ends here

# These are synchronous operations therefore taking too long to execute; Because database is huge; i need a concurrent solution
# activity_data = pd.read_sql(select_activity_stm, sql_engine)
# body_battery_activity_events_data = pd.read_sql(select_body_battery_activity_events_stm, sql_engine)
# body_battery_sample_data = pd.read_sql(select_body_battery_sample_stm, sql_engine)
# goals_data = pd.read_sql(select_goals_stm, sql_engine)
# heart_rate_sample_data = pd.read_sql(select_heart_rate_sample_stm, sql_engine)
# heart_rate_values_data = pd.read_sql(select_heart_rate_values_stm, sql_engine)
# kilocalories_data = pd.read_sql(select_kilocalories_stm, sql_engine)
# overall_sleep_data = pd.read_sql(select_overall_sleep_stm, sql_engine)
# sleep_data = pd.read_sql(select_sleep_stm, sql_engine)
# sleep_level_events_data = pd.read_sql(select_sleep_level_events_stm, sql_engine)
# steps_data = pd.read_sql(select_steps_stm, sql_engine)
# stress_data = pd.read_sql(select_stress_stm, sql_engine)
# stress_level_sample_data = pd.read_sql(select_stress_level_sample_stm, sql_engine)
# activity_data = pd.read_sql(select_activity_stm,sql_engine)
# goals_data = pd.read_sql(select_goals_stm, sql_engine)
# synchronous datapooling section ends here

# trying threadpools for data pooling

queries = {
    "activity_data": select_activity_stm,
    "body_battery_activity_events_data": select_body_battery_activity_events_stm,
    "body_battery_sample_data": select_body_battery_sample_stm,
    "goals_data": select_goals_stm,
    "heart_rate_sample_data": select_heart_rate_sample_stm,
    "heart_rate_values_data": select_heart_rate_values_stm,
    "kilocalories_data": select_kilocalories_stm,
    "overall_sleep_data": select_overall_sleep_stm,
    "sleep_data": select_sleep_stm,
    "sleep_level_events_data": select_sleep_level_events_stm,
    "steps_data": select_steps_stm,
    "stress_data": select_stress_stm,
    "stress_level_sample_data": select_stress_level_sample_stm,
}

def load_table(item):
    name, query = item

    with sql_engine.connect() as connection:
        df = pd.read_sql(query, connection)

    print(f"Loaded {name}: {len(df):,} rows")
    return name, df

with ThreadPoolExecutor(max_workers=4) as executor:
    results = executor.map(load_table, queries.items())

dataframes = dict(results)

# Access your DataFrames using the original variable names
activity_data = dataframes["activity_data"]
body_battery_activity_events_data = dataframes["body_battery_activity_events_data"]
body_battery_sample_data = dataframes["body_battery_sample_data"]
goals_data = dataframes["goals_data"]
heart_rate_sample_data = dataframes["heart_rate_sample_data"]
heart_rate_values_data = dataframes["heart_rate_values_data"]
kilocalories_data = dataframes["kilocalories_data"]
overall_sleep_data = dataframes["overall_sleep_data"]
sleep_data = dataframes["sleep_data"]
sleep_level_events_data = dataframes["sleep_level_events_data"]
steps_data = dataframes["steps_data"]
stress_data = dataframes["stress_data"]
stress_level_sample_data = dataframes["stress_level_sample_data"]




Loaded body_battery_activity_events_data: 448 rows
Loaded activity_data: 987 rows
Loaded goals_data: 987 rows
Loaded heart_rate_values_data: 961 rows
Loaded kilocalories_data: 987 rows
Loaded overall_sleep_data: 299 rows
Loaded sleep_data: 299 rows
Loaded steps_data: 987 rows
Loaded stress_data: 987 rows
Loaded sleep_level_events_data: 18,402 rows
Loaded body_battery_sample_data: 315,848 rows
Loaded stress_level_sample_data: 460,975 rows
Loaded heart_rate_sample_data: 4,912,004 rows


In [13]:
# Testing whether the concurrent  pooling worked and we have all the table with data available.
print("activity_data:", activity_data.shape)
print("body_battery_activity_events_data:", body_battery_activity_events_data.shape)
print("body_battery_sample_data:", body_battery_sample_data.shape)
print("goals_data:", goals_data.shape)
print("heart_rate_sample_data:", heart_rate_sample_data.shape)
print("heart_rate_values_data:", heart_rate_values_data.shape)
print("kilocalories_data:", kilocalories_data.shape)
print("overall_sleep_data:", overall_sleep_data.shape)
print("sleep_data:", sleep_data.shape)
print("sleep_level_events_data:", sleep_level_events_data.shape)
print("steps_data:", steps_data.shape)
print("stress_data:", stress_data.shape)
print("stress_level_sample_data:", stress_level_sample_data.shape)

activity_data: (987, 16)
body_battery_activity_events_data: (448, 14)
body_battery_sample_data: (315848, 12)
goals_data: (987, 13)
heart_rate_sample_data: (4912004, 12)
heart_rate_values_data: (961, 14)
kilocalories_data: (987, 12)
overall_sleep_data: (299, 12)
sleep_data: (299, 17)
sleep_level_events_data: (18402, 13)
steps_data: (987, 12)
stress_data: (987, 19)
stress_level_sample_data: (460975, 12)


In [14]:
# creating a database inventory to understand the database 

dataframes = {
    "activity_data": activity_data,
    "body_battery_activity_events_data": body_battery_activity_events_data,
    "body_battery_sample_data": body_battery_sample_data,
    "goals_data": goals_data,
    "heart_rate_sample_data": heart_rate_sample_data,
    "heart_rate_values_data": heart_rate_values_data,
    "kilocalories_data": kilocalories_data,
    "overall_sleep_data": overall_sleep_data,
    "sleep_data": sleep_data,
    "sleep_level_events_data": sleep_level_events_data,
    "steps_data": steps_data,
    "stress_data": stress_data,
    "stress_level_sample_data": stress_level_sample_data,
}

# 2. Create a database inventory
inventory = []

for name, df in dataframes.items():
    inventory.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "memory_MB": round(
            df.memory_usage(deep=True).sum() / 1024**2, 2
        ),
    })

inventory_df = pd.DataFrame(inventory)

print("DATABASE INVENTORY")
display(inventory_df)

# 3. Inspect column names, data types and example values
for name, df in dataframes.items():
    print(f"\n{'=' * 80}")
    print(f"TABLE: {name}")
    print(f"Shape: {df.shape}")

    schema = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(dtype) for dtype in df.dtypes],
        "missing_count": df.isna().sum().values,
        "missing_percent": (
            df.isna().mean().mul(100).round(2).values
        ),
        "unique_values": df.nunique(dropna=True).values,
    })

    display(schema)
    print("First 3 rows:")
    display(df.head(3))

# 4. Create a data-quality summary
quality = []

for name, df in dataframes.items():
    total_cells = df.size
    missing_cells = int(df.isna().sum().sum())

    quality.append({
        "table": name,
        "rows": len(df),
        "missing_cells": missing_cells,
        "missing_percent": round(
            missing_cells / total_cells * 100, 2
        ) if total_cells else 0,
        "duplicate_rows": int(df.duplicated().sum()),
    })

quality_df = pd.DataFrame(quality)

print("\nDATA QUALITY SUMMARY")
display(quality_df)

# 5. Summarise numerical columns
for name, df in dataframes.items():
    numeric = df.select_dtypes(include="number")

    if not numeric.empty:
        print(f"\nNUMERICAL SUMMARY: {name}")
        display(numeric.describe().T)


DATABASE INVENTORY


,table,rows,columns,memory_MB
0,activity_data,987,16,0.28
1,body_battery_activity_events_data,448,14,0.14
2,body_battery_sample_data,315848,12,78.57
3,goals_data,987,13,0.25
4,heart_rate_sample_data,4912004,12,1222.59
5,heart_rate_values_data,961,14,0.25
6,kilocalories_data,987,12,0.25
7,overall_sleep_data,299,12,0.09
8,sleep_data,299,17,0.19
9,sleep_level_events_data,18402,13,5.65



TABLE: activity_data
Shape: (987, 16)


,column,dtype,missing_count,missing_percent,unique_values
0,additional_id,str,0,0.00,987
1,sdv_user_id,int64,0,0.00,8
2,sdv_group_id,int64,0,0.00,3
3,sdv_data_type,str,0,0.00,1
4,calendar_date,object,0,0.00,251
5,start_measure_period_sec,int64,0,0.00,259
6,end_measure_period_sec,int64,0,0.00,272
7,duration_measure_period_sec,int64,0,0.00,17
8,week_nr,int64,0,0.00,37
9,week_day,int64,0,0.00,7


First 3 rows:


...

,count,mean,std,min,25%,50%,75%,max
sdv_user_id,987.0,4.200755e+03,2.156566e+02,3.724000e+03,4.274000e+03,4.286000e+03,4.306000e+03,4.408000e+03
sdv_group_id,987.0,7.553390e+03,5.202031e-01,7.552000e+03,7.553000e+03,7.553000e+03,7.554000e+03,7.554000e+03
start_measure_period_sec,987.0,1.779292e+09,5.253043e+06,1.767222e+09,1.776118e+09,1.779401e+09,1.783462e+09,1.788818e+09
end_measure_period_sec,987.0,1.779378e+09,5.252706e+06,1.767308e+09,1.776204e+09,1.779487e+09,1.783548e+09,1.788905e+09
duration_measure_period_sec,987.0,8.577684e+04,6.286301e+03,3.000000e+02,8.640000e+04,8.640000e+04,8.640000e+04,9.000000e+04
week_nr,987.0,2.095846e+01,8.694225e+00,1.000000e+00,1.600000e+01,2.100000e+01,2.800000e+01,3.700000e+01
week_day,987.0,4.020263e+00,1.990748e+00,1.000000e+00,2.000000e+00,4.000000e+00,6.000000e+00,7.000000e+00
steps,987.0,6.802246e+03,5.643962e+03,0.000000e+00,3.151000e+03,5.302000e+03,8.851500e+03,4.702500e+04
steps_goal,987.0,8.027082e+03,3.617580e+03,2.780000e+03,5.985000e+03,8.000000e+03,1.000000e+04,3.000000e+04



NUMERICAL SUMMARY: stress_data


,count,mean,std,min,25%,50%,75%,max
sdv_user_id,987.0,4.200755e+03,2.156566e+02,3.724000e+03,4.274000e+03,4.286000e+03,4.306000e+03,4.408000e+03
sdv_group_id,987.0,7.553390e+03,5.202031e-01,7.552000e+03,7.553000e+03,7.553000e+03,7.554000e+03,7.554000e+03
start_measure_period_sec,987.0,1.779292e+09,5.253043e+06,1.767222e+09,1.776118e+09,1.779401e+09,1.783462e+09,1.788818e+09
end_measure_period_sec,987.0,1.779378e+09,5.252706e+06,1.767308e+09,1.776204e+09,1.779487e+09,1.783548e+09,1.788905e+09
duration_measure_period_sec,987.0,8.577684e+04,6.286301e+03,3.000000e+02,8.640000e+04,8.640000e+04,8.640000e+04,9.000000e+04
week_nr,987.0,2.095846e+01,8.694225e+00,1.000000e+00,1.600000e+01,2.100000e+01,2.800000e+01,3.700000e+01
week_day,987.0,4.020263e+00,1.990748e+00,1.000000e+00,2.000000e+00,4.000000e+00,6.000000e+00,7.000000e+00
average_stress_level,987.0,3.243161e+01,1.227302e+01,-1.000000e+00,2.500000e+01,3.200000e+01,3.800000e+01,8.600000e+01
max_stress_level,960.0,9.436667e+01,6.449688e+00,3.200000e+01,9.300000e+01,9.600000e+01,9.900000e+01,1.000000e+02
stress_duration_sec,960.0,2.912775e+04,1.269687e+04,6.000000e+01,1.869000e+04,2.976000e+04,3.954000e+04,6.840000e+04



NUMERICAL SUMMARY: stress_level_sample_data


,count,mean,std,min,25%,50%,75%,max
sdv_user_id,460975.0,4.199134e+03,2.170695e+02,3.724000e+03,4.274000e+03,4.286000e+03,4.306000e+03,4.408000e+03
sdv_group_id,460975.0,7.553380e+03,5.174617e-01,7.552000e+03,7.553000e+03,7.553000e+03,7.554000e+03,7.554000e+03
start_measure_period_sec,460975.0,1.779330e+09,5.240222e+06,1.767222e+09,1.776118e+09,1.779401e+09,1.783548e+09,1.788818e+09
end_measure_period_sec,460975.0,1.779416e+09,5.240121e+06,1.767308e+09,1.776204e+09,1.779487e+09,1.783634e+09,1.788905e+09
duration_measure_period_sec,460975.0,8.622615e+04,2.830754e+03,3.000000e+02,8.640000e+04,8.640000e+04,8.640000e+04,9.000000e+04
week_nr,460975.0,2.102193e+01,8.675071e+00,1.000000e+00,1.600000e+01,2.100000e+01,2.800000e+01,3.700000e+01
week_day,460975.0,4.013734e+00,1.988962e+00,1.000000e+00,2.000000e+00,4.000000e+00,6.000000e+00,7.000000e+00
sample_point_sec,460975.0,1.779373e+09,5.240262e+06,1.767222e+09,1.776169e+09,1.779430e+09,1.783583e+09,1.788905e+09
stress_level,460975.0,2.160563e+01,2.401784e+01,-2.000000e+00,-1.000000e+00,1.700000e+01,3.400000e+01,1.000000e+02
